In [1]:
#!pip install uszipcode
#!pip install 'sqlalchemy_mate < 2.0.0.1'
#!pip install us
#!pip install pgeocode

In [2]:
import pandas as pd
from uszipcode import SearchEngine
import us

/opt/anaconda3/lib/python3.13/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
cdc_24 = pd.read_csv('../data/cleaned/cdc_cleaned_2024.csv')
cdc_24

,county,fips,deaths,Population
0,"Autauga County, AL",1001.0,50.0,408275
1,"Baldwin County, AL",1003.0,427.0,1671387
2,"Barbour County, AL",1005.0,33.0,172769
3,"Bibb County, AL",1007.0,49.0,155538
4,"Blount County, AL",1009.0,106.0,412077
...,...,...,...,...
2549,NaN,00nan,NaN,NaN
2550,NaN,00nan,NaN,NaN
2551,NaN,00nan,NaN,NaN
2552,NaN,00nan,NaN,NaN


In [4]:
cdc_24['deaths'].min()

10.0

In [5]:
facilities = pd.read_csv('../data/cleaned/samhsa_facilities_zip.csv')
facilities

,state,zip,facility_count
0,AK,99501,2
1,AK,99503,6
2,AK,99507,1
3,AK,99508,10
4,AK,99518,2
...,...,...,...
5344,WY,82941,1
5345,WY,83001,1
5346,WY,83101,1
5347,WY,83110,1


In [6]:
facilities['state'].eq('NJ').any()

np.False_

In [7]:
census_24 = pd.read_csv('../data/cleaned/census_acs_2024.csv')
census_24

,county_name,median_income,population,state,county,poverty_rate,unemployment_rate,fips
0,"Autauga County, Alabama",72481,59947,1,1,11.3,2.4,1001
1,"Baldwin County, Alabama",78775,246989,1,3,10.1,3.0,1003
2,"Barbour County, Alabama",46042,24643,1,5,21.4,7.8,1005
3,"Bibb County, Alabama",52541,22130,1,7,22.5,12.1,1007
4,"Blount County, Alabama",64190,59518,1,9,12.9,5.0,1009
...,...,...,...,...,...,...,...,...
3217,"Vega Baja Municipio, Puerto Rico",24244,53892,72,145,41.1,11.0,72145
3218,"Vieques Municipio, Puerto Rico",19803,8078,72,147,59.1,6.1,72147
3219,"Villalba Municipio, Puerto Rico",26286,21556,72,149,38.9,11.4,72149
3220,"Yabucoa Municipio, Puerto Rico",22944,29400,72,151,47.7,8.7,72151


## To begin this notebook I need to make sure that my county columns are able to be merged on for all of my datasets.

In [8]:
search = SearchEngine()

facilities["county"] = (
    facilities["zip"]
    .apply(
        lambda z:
        search.by_zipcode(z).county
        if search.by_zipcode(z)
        else None
    )
)

facilities.head()

,state,zip,facility_count,county
0,AK,99501,2,Anchorage Municipality
1,AK,99503,6,Anchorage Municipality
2,AK,99507,1,Anchorage Municipality
3,AK,99508,10,Anchorage Municipality
4,AK,99518,2,Anchorage Municipality


In [9]:
facilities.sample(5)

,state,zip,facility_count,county
3078,NC,28078,1,Mecklenburg County
3407,NY,10573,1,Westchester County
390,CA,90041,1,Los Angeles County
1926,IN,47501,2,Daviess County
3741,OH,44107,1,Cuyahoga County


In [10]:
facilities['county_state'] = facilities['county'] + ', ' + facilities['state']
facilities.head()

,state,zip,facility_count,county,county_state
0,AK,99501,2,Anchorage Municipality,"Anchorage Municipality, AK"
1,AK,99503,6,Anchorage Municipality,"Anchorage Municipality, AK"
2,AK,99507,1,Anchorage Municipality,"Anchorage Municipality, AK"
3,AK,99508,10,Anchorage Municipality,"Anchorage Municipality, AK"
4,AK,99518,2,Anchorage Municipality,"Anchorage Municipality, AK"


In [11]:
facilities = facilities.groupby('county_state')['facility_count'].sum().reset_index()
facilities

,county_state,facility_count
0,", GU",1
1,", MP",1
2,"Acadia Parish, LA",1
3,"Accomack County, VA",2
4,"Ada County, ID",36
...,...,...
2082,"Yuba County, CA",4
2083,"Yukon-Koyukuk Census Area, AK",1
2084,"Yuma County, AZ",15
2085,"Yuma County, CO",1


In [12]:
cdc_24["fips"] = (
    cdc_24["fips"]
    .astype(str)
    .str.zfill(5)
)

census_24["fips"] = (
    census_24["fips"]
    .astype(str)
    .str.zfill(5)
)

In [13]:
#rename state and county column to mergre on
cdc_24 = cdc_24.rename(columns= {'county': 'county_state'})
cdc_24.head(2)

,county_state,fips,deaths,Population
0,"Autauga County, AL",1001.0,50.0,408275
1,"Baldwin County, AL",1003.0,427.0,1671387


In [14]:
census_24 = census_24.rename(columns= {'county_name': 'county_state'})
census_24.head(2)

,county_state,median_income,population,state,county,poverty_rate,unemployment_rate,fips
0,"Autauga County, Alabama",72481,59947,1,1,11.3,2.4,01001
1,"Baldwin County, Alabama",78775,246989,1,3,10.1,3.0,01003


In [15]:
def safe_state_lookup(state_name):
    """Safely lookup state abbreviation, return original name if not found"""
    try:
        state = us.states.lookup(state_name.strip())  
        return state.abbr if state else state_name  
    except:
        return state_name 

census_24['county_state'] = (
    census_24['county_state']
    .str.split(', ')
    .apply(lambda x: f"{x[0]}, {safe_state_lookup(x[1])}")
)

census_24.head(2)

,county_state,median_income,population,state,county,poverty_rate,unemployment_rate,fips
0,"Autauga County, AL",72481,59947,1,1,11.3,2.4,01001
1,"Baldwin County, AL",78775,246989,1,3,10.1,3.0,01003


In [16]:
census_24['county_state'].value_counts()

county_state
Autauga County, AL     1
Custer County, OK      1
Carter County, OK      1
Cherokee County, OK    1
Choctaw County, OK     1
                      ..
Meade County, KY       1
Menifee County, KY     1
Mercer County, KY      1
Metcalfe County, KY    1
Yauco Municipio, PR    1
Name: count, Length: 3222, dtype: int64

In [17]:
facilities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2087 entries, 0 to 2086
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   county_state    2087 non-null   object
 1   facility_count  2087 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 32.7+ KB


## Merging 

In [18]:
facilities.head(2)

,county_state,facility_count
0,", GU",1
1,", MP",1


In [19]:
master_raw = cdc_24.merge(census_24, on='county_state', how= 'outer') \
               .merge(facilities, on='county_state', how= 'left')
master_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3314 entries, 0 to 3313
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   county_state       3242 non-null   object 
 1   fips_x             2554 non-null   object 
 2   deaths             2483 non-null   float64
 3   Population         2483 non-null   object 
 4   median_income      3222 non-null   float64
 5   population         3222 non-null   float64
 6   state              3222 non-null   float64
 7   county             3222 non-null   float64
 8   poverty_rate       3222 non-null   float64
 9   unemployment_rate  3222 non-null   float64
 10  fips_y             3222 non-null   object 
 11  facility_count     2082 non-null   float64
dtypes: float64(8), object(4)
memory usage: 310.8+ KB


In [20]:
master_raw.shape

(3314, 12)

In [21]:
master_raw.head(2)

,county_state,fips_x,deaths,Population,median_income,population,state,county,poverty_rate,unemployment_rate,fips_y,facility_count
0,"Abbeville County, SC",45001.0,45.0,171168,52935.0,24420.0,45.0,1.0,15.7,4.0,45001,NaN
1,"Acadia Parish, LA",22001.0,151.0,413278,45562.0,56955.0,22.0,1.0,25.0,7.7,22001,1.0


In [22]:
master_raw['county_state'] 

0       Abbeville County, SC
1          Acadia Parish, LA
2        Accomack County, VA
3             Ada County, ID
4           Adair County, IA
                ...         
3309                     NaN
3310                     NaN
3311                     NaN
3312                     NaN
3313                     NaN
Name: county_state, Length: 3314, dtype: object

### Now that I have my raw master data frame I can clean it up

In [23]:
master_raw = master_raw.drop(columns=['state', 'county', 'fips_y'])

In [24]:
master_raw.head(2)

,county_state,fips_x,deaths,Population,median_income,population,poverty_rate,unemployment_rate,facility_count
0,"Abbeville County, SC",45001.0,45.0,171168,52935.0,24420.0,15.7,4.0,NaN
1,"Acadia Parish, LA",22001.0,151.0,413278,45562.0,56955.0,25.0,7.7,1.0


In [25]:
master = master_raw.rename(columns={
    'fips_x': 'fips'
})


In [26]:
#last clean up 
master['fips'] = (
    master['fips']
    .replace('00nan', None)
    .fillna('00000')
)

In [27]:
master_raw['county_state'].str.contains('NJ').any()

np.True_

In [28]:
master.head(2)

,county_state,fips,deaths,Population,median_income,population,poverty_rate,unemployment_rate,facility_count
0,"Abbeville County, SC",45001.0,45.0,171168,52935.0,24420.0,15.7,4.0,NaN
1,"Acadia Parish, LA",22001.0,151.0,413278,45562.0,56955.0,25.0,7.7,1.0


In [29]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3314 entries, 0 to 3313
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   county_state       3242 non-null   object 
 1   fips               3314 non-null   object 
 2   deaths             2483 non-null   float64
 3   Population         2483 non-null   object 
 4   median_income      3222 non-null   float64
 5   population         3222 non-null   float64
 6   poverty_rate       3222 non-null   float64
 7   unemployment_rate  3222 non-null   float64
 8   facility_count     2082 non-null   float64
dtypes: float64(6), object(3)
memory usage: 233.1+ KB


In [30]:
master.shape

(3314, 9)

## Now that I have my Master Data Frame, will add come variables that I plan to work with.

In [31]:
master.columns[master.columns.duplicated()]

Index([], dtype='object')

In [32]:
#analysis variable
master["overdose_rate"] = (
    master["deaths"]
    /
    master["population"]
) * 100000

In [33]:
master["facility_rate"] = (
    master["facility_count"]
    /
    master["population"]
) * 100000

In [34]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3314 entries, 0 to 3313
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   county_state       3242 non-null   object 
 1   fips               3314 non-null   object 
 2   deaths             2483 non-null   float64
 3   Population         2483 non-null   object 
 4   median_income      3222 non-null   float64
 5   population         3222 non-null   float64
 6   poverty_rate       3222 non-null   float64
 7   unemployment_rate  3222 non-null   float64
 8   facility_count     2082 non-null   float64
 9   overdose_rate      2462 non-null   float64
 10  facility_rate      2079 non-null   float64
dtypes: float64(8), object(3)
memory usage: 284.9+ KB


Now lets save it as a csv

In [35]:
master.to_csv('../data/master_df.csv', index=False)

In [36]:
# ── Merge USDA Rural-Urban Continuum Codes and export enriched CSV ─────────
import requests
from io import BytesIO

print('Downloading USDA RUCC data...')
rucc_url = 'https://www.ers.usda.gov/media/5767/2023-rural-urban-continuum-codes.xlsx'
response = requests.get(rucc_url)
rucc_raw = pd.read_excel(BytesIO(response.content))

# Auto-detect column names
rucc_col = [c for c in rucc_raw.columns if 'RUCC' in c.upper()][0]
fips_col = [c for c in rucc_raw.columns if 'FIPS' in c.upper()][0]
desc_col  = [c for c in rucc_raw.columns if 'DESC' in c.upper() or 'DESCRIPTION' in c.upper()]

keep = [fips_col, rucc_col] + (desc_col[:1] if desc_col else [])
rucc = rucc_raw[keep].copy()
rucc.columns = ['fips', 'rucc'] + (['rucc_desc'] if desc_col else [])
rucc['fips'] = rucc['fips'].astype(str).str.zfill(5)

print(f'RUCC rows: {len(rucc):,}')
print(f'RUCC distribution:\n{rucc["rucc"].value_counts().sort_index()}')

RUCC rows: 3,235
RUCC distribution:
rucc
1.0    483
2.0    398
3.0    371
4.0    204
5.0     81
6.0    385
7.0    248
8.0    468
9.0    595
Name: count, dtype: int64


In [37]:
# ── Fix master FIPS and merge ──────────────────────────────────────────────
master_export = master.copy()

# Fix FIPS: drop .0 float artifact, zero-pad to 5 digits
master_export['fips'] = (
    master_export['fips']
    .replace('00000', None)          # treat bad FIPS as null
    .dropna()
    .pipe(lambda s: s.loc[s.index])  # keep index alignment
)
master_export['fips'] = pd.to_numeric(master_export['fips'], errors='coerce')
master_export = master_export.dropna(subset=['fips'])
master_export['fips'] = master_export['fips'].astype(int).astype(str).str.zfill(5)

# Merge RUCC
master_rucc = master_export.merge(rucc[['fips', 'rucc']], on='fips', how='left')
master_rucc['is_rural'] = (master_rucc['rucc'] >= 4).astype('Int64')

# Deduplicate — one row per county
master_rucc = master_rucc.drop_duplicates(subset='fips', keep='first')

print(f'Rows: {len(master_rucc):,}')
print(f'Unique FIPS: {master_rucc["fips"].nunique():,}')
print(f'Counties with RUCC: {master_rucc["rucc"].notna().sum():,}')

# Export
master_rucc.to_csv('../data/master_df_rucc.csv', index=False)
print('✅ Saved: ../data/master_df_rucc.csv')

Rows: 2,482
Unique FIPS: 2,482
Counties with RUCC: 2,471
✅ Saved: ../data/master_df_rucc.csv
